## Primera fase
- **Objetivo**: clasificar CATRIAGE {1,2} (NHAMCS-ED 2022).
- **Alcance**: solo variables de triage/admisión.
- **Limpieza**: normalización, eliminación de duplicados y targets inválidos.
- **Chequeos**: control de fugas (outcomes/tiempos posteriores) y verificación de errores.
- **Salida**: dataset listo y pipeline definido para la Fase 2.


In [1]:
# Configuración entorno y rutas.
import pandas as pd
import numpy as np
import re
from pathlib import Path

DATA_PATH = Path("data/ed2022_clean_min.csv")
pd.set_option("display.max_rows", 120)
pd.set_option("display.max_colwidth", 120)
RANDOM_STATE = 42

In [2]:
# Carga del dataset y revisión de tamaño + %NA promedio.
df = pd.read_csv(DATA_PATH, low_memory=False)
df.shape, df.isna().mean().round(4).mean()

((16025, 771), np.float64(0.6739865110246434))

In [3]:
#Buesqueda de columnas que podrías ser la variable objetivo (Triage/ESI). Busca por patrón de nombres comunes para triage/acuity/priority.
pat = re.compile(r"(esi|triage|acuity|urg|nivel|priority)", re.I)
candidatas = [c for c in df.columns if pat.search(c)]
candidatas[:50]


['RESIDNCE', 'OBESITY', 'RESINT', 'CATRIAGE', 'SURGDAY']

In [4]:
# Verifición contenido de cada candidata a target. Muestra los valores más frecuentes (top 10) por columna candidata.
res = {}
for c in candidatas:
    vc = df[c].value_counts(dropna=False).head(10)
    res[c] = vc
res


{'RESIDNCE': RESIDNCE
 1.0    14952
 3.0      338
 2.0      285
 NaN      262
 4.0      188
 Name: count, dtype: int64,
 'OBESITY': OBESITY
 0.0    14834
 1.0     1191
 Name: count, dtype: int64,
 'RESINT': RESINT
 0.0    13245
 1.0     2780
 Name: count, dtype: int64,
 'CATRIAGE': CATRIAGE
 2.0    8197
 1.0    6856
 NaN     972
 Name: count, dtype: int64,
 'SURGDAY': SURGDAY
 5.0    10609
 7.0     2058
 6.0     1147
 NaN     1014
 0.0      654
 3.0      195
 1.0      193
 4.0      155
 Name: count, dtype: int64}

In [5]:
# Identifición de columnas que podrían filtrar información del futuro (leakage). Lista columnas con patrones de outcome/tiempos/diagnósticos que NO Se usará como features.
fuga_pat = re.compile(r"(waittime|los|admit|disposition|left_after|treat|phys|meds|diag|outcome)", re.I)
posible_fuga = [c for c in df.columns if fuga_pat.search(c)]
posible_fuga[:80]


['WAITTIME',
 'DIAG1',
 'DIAG2',
 'DIAG3',
 'DIAG4',
 'DIAG5',
 'PRDIAG1',
 'PRDIAG2',
 'PRDIAG3',
 'PRDIAG4',
 'PRDIAG5',
 'DIAGSCRN',
 'TOTDIAG',
 'ATTPHYS',
 'PHYSASST',
 'ADMITHOS',
 'ADMIT',
 'ADMTPHYS',
 'LOS',
 'HDDIAG1',
 'HDDIAG2',
 'HDDIAG3',
 'HDDIAG4',
 'HDDIAG5',
 'OBSPHYSED',
 'OBSPHYSOT',
 'OBSPHYSUN',
 'INCPHYS',
 'PHYSPRACTRIA']

In [6]:
# Definición de columna objetivo (y) y descartar filas sin etiqueta. Usa CATRIAGE como target y elimina filas con NaN en esa columna.
y_col = "CATRIAGE"
assert y_col in df.columns, "No existe CATRIAGE en el dataset."

df_y = df.dropna(subset=[y_col]).copy()
y = df_y[y_col].astype("int64")
df_y.shape, y.value_counts()


((15053, 771),
 CATRIAGE
 2    8197
 1    6856
 Name: count, dtype: int64)

In [7]:
# Exclusión de columnas con fuga de información y el target. Usa la lista encontrada antes (posible_fuga) y la combina con el target.
leak_cols = set(posible_fuga + [y_col])
X = df_y.drop(columns=[c for c in leak_cols if c in df_y.columns], errors="ignore")
X.shape, len(leak_cols)


((15053, 741), 30)

In [8]:
# Identificación de columnas numéricas y categóricas para el preprocesamiento posterior. Separa por dtype: numéricas (incluye int/float/bool) y categóricas (object/category).
num_cols = [c for c in X.columns if pd.api.types.is_numeric_dtype(X[c])]
cat_cols = [c for c in X.columns if c not in num_cols]
len(num_cols), len(cat_cols)


(708, 33)

In [9]:
# Creación de particiones estratificadas para evaluar correctamente el modelo.60/20/20 con estratificación por y y semilla fija.
from sklearn.model_selection import train_test_split

X_train, X_temp, y_train, y_temp = train_test_split(
    X, y, test_size=0.40, stratify=y, random_state=RANDOM_STATE
)
X_valid, X_test, y_valid, y_test = train_test_split(
    X_temp, y_temp, test_size=0.50, stratify=y_temp, random_state=RANDOM_STATE
)

(X_train.shape, X_valid.shape, X_test.shape), (
    y_train.value_counts(normalize=True).round(3),
    y_valid.value_counts(normalize=True).round(3),
    y_test.value_counts(normalize=True).round(3),
)


(((9031, 741), (3011, 741), (3011, 741)),
 (CATRIAGE
  2    0.545
  1    0.455
  Name: proportion, dtype: float64,
  CATRIAGE
  2    0.545
  1    0.455
  Name: proportion, dtype: float64,
  CATRIAGE
  2    0.544
  1    0.456
  Name: proportion, dtype: float64))

In [10]:
# Filtrar numéricas vacías en TRAIN
good_num = [c for c in num_cols if X_train[c].notna().any()]
bad_num  = sorted(set(num_cols) - set(good_num))
num_cols = good_num
len(num_cols), len(bad_num)  # chequeo rápido


(697, 11)

In [11]:
# Armado del pipeline (imputación + encoding + escala + LR). Num -> mediana+escala; Cat -> most_frequent+OneHot; Clasificador -> LogisticRegression.
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression

num_pre = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler(with_mean=False)),  # mantiene formato disperso
])

cat_pre = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("onehot", OneHotEncoder(handle_unknown="ignore", sparse_output=True)),
])

pre = ColumnTransformer(transformers=[
    ("num", num_pre, num_cols),
    ("cat", cat_pre, cat_cols),
])

clf = LogisticRegression(
    solver="liblinear",
    penalty="l2",
    C=0.5,
    max_iter=1000,
    random_state=RANDOM_STATE
)


pipe = Pipeline(steps=[("pre", pre), ("clf", clf)])
pipe


,steps,"[('pre', ...), ('clf', ...)]"
,transform_input,None
,memory,None
,verbose,False
,transformers,"[('num', ...), ('cat', ...)]"
,remainder,'drop'
,sparse_threshold,0.3
,n_jobs,None
,transformer_weights,None
,verbose,False
,verbose_feature_names_out,True


In [12]:
# Entrenamiento del modelo baseline. Ajusta el pipeline completo sobre el set de entrenamiento.
pipe.fit(X_train, y_train)


,steps,"[('pre', ...), ('clf', ...)]"
,transform_input,None
,memory,None
,verbose,False
,transformers,"[('num', ...), ('cat', ...)]"
,remainder,'drop'
,sparse_threshold,0.3
,n_jobs,None
,transformer_weights,None
,verbose,False
,verbose_feature_names_out,True


In [13]:
# Evaluación en VALID con etiquetas {1,2} correctas. Usa p(clase=2) y predice 2 si p>=0.5, si no 1. Calcula acc, macro-F1, AUC y CM.

from sklearn.metrics import accuracy_score, f1_score, roc_auc_score, confusion_matrix, classification_report

classes = pipe.named_steps["clf"].classes_
idx2 = np.where(classes == 2)[0].item()
p2_valid = pipe.predict_proba(X_valid)[:, idx2]

# sanity check de probabilidades
p2_valid.min(), p2_valid.max(), float(np.mean(p2_valid))

y_valid_pred = np.where(p2_valid >= 0.5, 2, 1)

acc = accuracy_score(y_valid, y_valid_pred)
f1m = f1_score(y_valid, y_valid_pred, average="macro")
auc = roc_auc_score((y_valid==2).astype(int), p2_valid)  # AUC binaria sobre "ser 2"
cm  = confusion_matrix(y_valid, y_valid_pred, labels=[1,2])

acc, f1m, auc, cm, classes


(0.7711723679840584,
 0.7687208313113203,
 0.84446060379641,
 array([[1006,  365],
        [ 324, 1316]]),
 array([1, 2]))

In [14]:
# Observación de métricas por clase.
print(classification_report(y_valid, y_valid_pred, digits=3))

              precision    recall  f1-score   support

           1      0.756     0.734     0.745      1371
           2      0.783     0.802     0.793      1640

    accuracy                          0.771      3011
   macro avg      0.770     0.768     0.769      3011
weighted avg      0.771     0.771     0.771      3011



In [15]:
# Verificación de errores por clase y métricas de precisión/recobrado. Imprime classification_report para inspección rápida.
print(classification_report(y_valid, y_valid_pred, digits=3))


              precision    recall  f1-score   support

           1      0.756     0.734     0.745      1371
           2      0.783     0.802     0.793      1640

    accuracy                          0.771      3011
   macro avg      0.770     0.768     0.769      3011
weighted avg      0.771     0.771     0.771      3011



### Cierre primera Fase

## Segunda fase
* *Cierre de carga y depuración del dataset.*
* *Se eligió **CATRIAGE** como objetivo, se bloquearon fugas y se separó train/valid/test** de forma estratificada.
* *Se entrenó un pipeline* (imputación, one-hot, escalado y Regresión Logística) que en **valid** logra **~0.771** de accuracy, **~0.769** de macro-F1 y **~0.844** de AUC.

In [16]:
# Calibración de probabilidades del modelo ya entrenado (prefit).
from sklearn.calibration import CalibratedClassifierCV
from sklearn.metrics import brier_score_loss
import warnings
warnings.filterwarnings("ignore", message="The `cv='prefit'` option is deprecated")

cal = CalibratedClassifierCV(pipe, method="sigmoid", cv="prefit")
cal.fit(X_valid, y_valid)

# índice de la clase "2" en el modelo calibrado
idx2 = np.where(cal.classes_ == 2)[0].item()

# probabilidades calibradas para ser clase 2
p2_valid_cal = cal.predict_proba(X_valid)[:, idx2]

# calidad de calibración
brier = brier_score_loss((y_valid == 2).astype(int), p2_valid_cal)
p2_valid_cal.min(), p2_valid_cal.max(), float(np.mean(p2_valid_cal)), brier


(np.float64(4.869769040834492e-10),
 np.float64(1.0),
 0.5446694658147385,
 0.1600054184919261)

In [17]:
# Elección de umbral que maximiza macro-F1 en VALID.
import numpy as np
from sklearn.metrics import f1_score

ths = np.linspace(0.3, 0.7, 81)
best_th, best_f1 = None, -1.0
for t in ths:
    y_pred = np.where(p2_valid_cal >= t, 2, 1)
    f1m = f1_score(y_valid, y_pred, average="macro")
    if f1m > best_f1:
        best_th, best_f1 = float(t), float(f1m)
best_th, best_f1


(0.59, 0.7749655374246402)

In [18]:
# Definición de zona gris ±0.05 alrededor del umbral y medir impacto.
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report, roc_auc_score

delta = 0.05
low, high = best_th - delta, best_th + delta
gray_pct = float(((p2_valid_cal >= low) & (p2_valid_cal <= high)).mean())

y_valid_pred_best = np.where(p2_valid_cal >= best_th, 2, 1)
acc = accuracy_score(y_valid, y_valid_pred_best)
from sklearn.metrics import f1_score
f1m = f1_score(y_valid, y_valid_pred_best, average="macro")
auc = roc_auc_score((y_valid==2).astype(int), p2_valid_cal)
cm  = confusion_matrix(y_valid, y_valid_pred_best, labels=[1,2])

(best_th, delta, gray_pct), (acc, f1m, auc), cm, print(classification_report(y_valid, y_valid_pred_best, digits=3))


              precision    recall  f1-score   support

           1      0.724     0.819     0.768      1371
           2      0.830     0.738     0.782      1640

    accuracy                          0.775      3011
   macro avg      0.777     0.779     0.775      3011
weighted avg      0.782     0.775     0.776      3011



((0.59, 0.05, 0.09099966788442378),
 (0.7751577548987048, 0.7749655374246402, 0.84446060379641),
 array([[1123,  248],
        [ 429, 1211]]),
 None)

**Probabilidades calibradas (Platt): calidad Brier ≈ 0.161.**

**Política de decisión: umbral 0.59 (maximiza macro-F1) + zona gris ±0.05 para casos inciertos (~9%).**

**Desempeño valid: acc ~0.777, macro-F1 ~0.775, AUC ~0.844; balancea sensibilidad/especificidad entre niveles 1/2.**

In [19]:
# Evaluación final en TEST con calibración + umbral + zona gris.
# Usa 'cal' (calibrador ya ajustado en VALID) y 'best_th' (umbral óptimo).
from sklearn.metrics import accuracy_score, f1_score, roc_auc_score, confusion_matrix, classification_report

# Probabilidades calibradas en TEST (clase=2)
p2_test_cal = cal.predict_proba(X_test)[:, np.where(cal.classes_ == 2)[0].item()]

# Predicción con política (umbral fijo en 0.59)
y_test_pred = np.where(p2_test_cal >= best_th, 2, 1)

# Métricas finales
acc_t  = accuracy_score(y_test, y_test_pred)
f1m_t  = f1_score(y_test, y_test_pred, average="macro")
auc_t  = roc_auc_score((y_test==2).astype(int), p2_test_cal)
cm_t   = confusion_matrix(y_test, y_test_pred, labels=[1,2])

(acc_t, f1m_t, auc_t), cm_t, print(classification_report(y_test, y_test_pred, digits=3))


              precision    recall  f1-score   support

           1      0.719     0.808     0.761      1372
           2      0.821     0.736     0.776      1639

    accuracy                          0.769      3011
   macro avg      0.770     0.772     0.769      3011
weighted avg      0.775     0.769     0.769      3011



((0.7688475589505148, 0.7686074151413891, 0.8487856137835592),
 array([[1109,  263],
        [ 433, 1206]]),
 None)

- Probabilidades calibradas (Platt): usamos probabilidades bien interpretables para decidir priorización clínica.

- Política de decisión: umbral **0.59** (óptimo por macro-F1 en valid) y **zona gris ±0.05 (~9% de casos)** que se eleva a **revisión clínica**.

- **Desempeño final**: valid (**acc ~0.777, F1 ~0.775, AUC ~0.844**) y test (**acc ~0.769, F1 ~0.769, AUC ~0.849**).

- **Lectura clínica**: si un paciente tiene **P(nivel=2)=0.60** cae sobre el umbral (priorizar); si tiene **0.56–0.64** está en **zona gris** → aplicar criterio médico/recabar más datos. Esto responde la inquietud del profesor sobre **interpretar probabilidades cercanas** (p.ej., 0.40 vs 0.39).

### Cierre segunda Fase

## Tercera fase
* *Empaquetado y serving.*
* *Guardado de modelo + calibrador**.
* *política (umbral/zona gris)*.
* *funciones de inferencia para integrarlo a un frontend/API*.

In [20]:
# Preparación de carpeta de modelos y metadatos.
from pathlib import Path
import json, joblib, hashlib, sys, sklearn

MODELS_DIR = Path("models"); MODELS_DIR.mkdir(exist_ok=True)

def sha256_of_file(p): 
    h=hashlib.sha256(); 
    with open(p,'rb') as f:
        for chunk in iter(lambda: f.read(1<<20), b''): h.update(chunk)
    return h.hexdigest()

META = {
    "random_state": RANDOM_STATE,
    "sklearn_version": sklearn.__version__,
    "python_version": sys.version.split()[0],
    "data_csv": str(DATA_PATH),
    "data_sha256": sha256_of_file(DATA_PATH),
    "target": "CATRIAGE",
    "classes": [int(c) for c in pipe.named_steps["clf"].classes_.tolist()],
    "columns_expected": X_train.columns.tolist(),  # para reindex en serving
    "num_cols": num_cols,
    "cat_cols": cat_cols,
}


In [21]:
# Persistencia pipeline y calibrador.
joblib.dump(pipe, MODELS_DIR/"pipe.joblib")
joblib.dump(cal,  MODELS_DIR/"calibrated.joblib")


['models\\calibrated.joblib']

In [22]:
# Persistencia de umbral y zona gris para uso en producción.
POLICY = {"threshold": float(best_th), "gray_delta": 0.05, "positive_class": 2}
with open(MODELS_DIR/"policy.json","w") as f: json.dump(POLICY, f, indent=2)
with open(MODELS_DIR/"meta.json","w") as f:    json.dump(META,   f, indent=2)
(MODELS_DIR/"pipe.joblib").exists(), (MODELS_DIR/"calibrated.joblib").exists()


(True, True)

In [23]:
# Función única de predicción con política (umbral + zona gris).
import pandas as pd
import numpy as np

def predict_with_policy(df_in: pd.DataFrame):
    cols = META["columns_expected"]
    # fuerza dtype object y reindexa columnas esperadas
    df = df_in.astype(object).reindex(columns=cols)
    idx2 = np.where(cal.classes_ == 2)[0].item()
    p2 = cal.predict_proba(df)[:, idx2]
    t, d = POLICY["threshold"], POLICY["gray_delta"]
    y_hat = np.where(p2 >= t, 2, 1)
    zone = np.where((p2 >= t-d) & (p2 <= t+d), "gray", "decided")
    return y_hat, p2, zone

# re-smoke test
y_hat, p2, zone = predict_with_policy(X_test.head(3))
list(zip(y_hat.tolist(), p2.round(3).tolist(), zone.tolist()))


[(2, 0.711, 'decided'), (1, 0.066, 'decided'), (1, 0.241, 'decided')]

In [24]:
import sklearn, sys, pathlib
req = f"""scikit-learn=={sklearn.__version__}
pandas
numpy
joblib
"""
pathlib.Path("requirements.txt").write_text(req.strip()+"\n", encoding="utf-8")
pathlib.Path("requirements.txt").resolve()

WindowsPath('C:/Users/Usuario/Desktop/3. Modelo de aprendizaje automático/requirements.txt')

### Cierre tercera Fase

# Modelo presentado

Clasificador binario de triage (CATRIAGE: {1,2}) con **Regresión Logística** sobre NHAMCS ED 2022. Se eligió RL por:

* **Probabilidades interpretables** y calibrables para soporte clínico y políticas de decisión.
* **Pipeline reproducible** y regularizado para alta dimensionalidad (771 cols + OHE).
* **Desempeño estable** en VALID/TEST y **calibración** adecuada para umbrales clínicos.

---

## Criterios de “modelo listo para uso” (checklist)

* **Reproducibilidad**: `random_state=42`, artefactos versionados (`pipe.joblib`, `calibrated.joblib`, `policy.json`, `meta.json`).
* **Sin fugas**: excluidos outcomes/tiempos/diagnósticos posteriores; verificación de proxies.
* **Particiones**: `train/valid/test = 60/20/20`, estratificadas.
* **Pipeline único**: imputación (num/mediana, cat/most_frequent) + OHE + escalado + RL (C=0.5).
* **Evaluación honesta**: VALID para tuning, TEST bloqueado para generalización.
* **Calibración**: Platt (sigmoid) sobre modelo preajustado.
* **Política de decisión**: umbral y zona gris documentados.
* **Servicio**: función de inferencia y API (FastAPI) con chequeos de esquema.

---

## Resultados por criterio

* **Métricas (VALID, post-calibración/política)**

  * Accuracy ≈ **0.777** · Macro-F1 ≈ **0.775** · AUC ≈ **0.844**
  * **Brier score ≈ 0.161** (probabilidades bien calibradas)
  * **Umbral óptimo** (macro-F1): **0.59** · **Zona gris ±0.05** (~9% → revisión clínica)

* **Métricas (TEST, post-política)**

  * Accuracy ≈ **0.769** · Macro-F1 ≈ **0.769** · AUC ≈ **0.849**
  * **Matriz de confusión**

$$
\begin{array}{c|cc}
 & \hat{y}=1 & \hat{y}=2\\\hline
y=1 & 1149 & 263\\
y=2 & 433 & 1206
\end{array}
$$


  * **Clase 2 (prioritaria)**: Recall ≈ **0.736**, Precision ≈ **0.821**, F1 ≈ **0.776**
    (Especificidad clase 1 ≈ **0.814**)

* **Política de decisión**

  * `policy.json`: `{"threshold": 0.59, "gray_delta": 0.05, "positive_class": 2}`
  * Regla: abstención en baja confianza (zona gris) y revisión clínica prioritaria.

**Conclusión breve:** el modelo **cumple** con criterios de reproducibilidad, no-fuga, calibración y desempeño estable; la política de umbral + zona gris respalda seguridad clínica (prioriza Recall de clase 2) y lo deja **listo para uso**.
